# Data sources 6/6: repositories and organizations of a user

`user_repos`, `user_orgs`, `org_repos` list them as plain dicts; `clone_user_repos` and
`clone_org_repos` clone them into `<path>/<name>`. With the user's own token, private
repositories (`privacy="all"`) and every organization membership show up.

In [ ]:
from gitrecon.config import load_env
from gitrecon.sources import repos

load_env()
mine = repos.user_repos("sarverott", include_forks=False)
print(len(mine), "own repositories without forks,", round(sum(r["size_kib"] for r in mine) / 1024, 1), "MiB")
mine[0]

In [ ]:
orgs = repos.user_orgs("sarverott")
for org in orgs:
    listed = repos.org_repos(org["login"])
    print(f"{org['login']:<24} {len(listed):>4} repos  {sum(r['size_kib'] for r in listed) / 1024:>9.1f} MiB  "
          f"forks {sum(r['fork'] for r in listed)}")

Some organizations refuse classic tokens; the client then repeats the request without one
(public data only) and says so in the log.

## Cloning

Two of them, shallow (`depth=1`: the newest commit only), into a temporary folder. The real
targets are below, switched off - check sizes first: forks make most of the weight.

In [ ]:
import tempfile

repos.clone_repos(mine[:2], tempfile.mkdtemp(prefix="repos-"), depth=1)

In [ ]:
CLONE_ALL = False
if CLONE_ALL:
    repos.clone_user_repos("sarverott", "~/__WORKSHOP/forge/sarverott/repos", include_forks=False, progress=print)
    for org in orgs:
        repos.clone_org_repos(org["login"], f"~/__WORKSHOP/forge/{org['login']}", include_forks=False, progress=print)

CLI: `gitrecon repos sarverott`, `gitrecon orgs sarverott`, `gitrecon org-repos The-Apokryf`,
`gitrecon repo-clone sarverott PATH --dry-run`, `gitrecon org-clone The-Apokryf PATH --no-forks --depth 1`.